In [3]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / 'src'))

from fetch_trivsel_fravaer import fetch, QUERY_FRAVAER_TIME, QUERY_TRIVSEL_TIME, QUERY_FORAELDRE_TIME, to_float


import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
import numpy as np

In [4]:
df_trivsel_time = fetch(QUERY_TRIVSEL_TIME, "Trivsel")
df_fravaer_time = fetch(QUERY_FRAVAER_TIME, "Fravær")
df_foraeldre_time = fetch(QUERY_FORAELDRE_TIME, "Forældres uddannelse")

Trivsel: hentede 9238 rækker
Fravær: hentede 2311 rækker
Forældres uddannelse: hentede 27137 rækker


Gentag processen fra tidligere med at rense data og merge de to trivsels- og fraværsdatasæt. 

In [5]:
display(df_trivsel_time.head(1))
display(df_fravaer_time.head(1))
display(df_foraeldre_time.head(1))

,[Institution].[Beliggenhedskommune].[Beliggenhedskommune],[Klassetrin].[Klassetringruppe].[Klassetringruppe],[Skoleår].[Skoleår].[Skoleår],[Trivselsindikator].[Trivselsindikator].[Trivselsindikator],[Trivselsindikator Svarintervaller].[Trivselsindikator Svarintervaller].[Trivselsindikator Svarintervaller],Andel Indikatorsvar
0,Albertslund,Udskoling,2014/2015,Generel trivsel,"4,1 til 5","14,8 %"


,[Institution].[Beliggenhedskommune].[Beliggenhedskommune],[Klassetrin].[Skoletrin].[Skoletrin],[Tid].[Skoleår].[Skoleår],Gennemsnitligt fravær per skoleår
0,Albertslund,10. klasse mv.,2019/2020,"20,1 %"


,[Forældres Højest Fuldførte Uddannelse].[Uddannelsesgruppe].[Uddannelsesgruppe],[Institution].[Beliggenhedskommune].[Beliggenhedskommune],[Klassetrin].[Klassetringruppe].[Klassetringruppe],[Skoleår].[Skoleår].[Skoleår],Antal elever
0,<fejl>,Assens,Indskoling,2018/2019,5


In [6]:
#Omdøb kolonner
df_trivsel_time = df_trivsel_time.rename(columns={
    '[Institution].[Beliggenhedskommune].[Beliggenhedskommune]': 'kommune',
    '[Klassetrin].[Klassetringruppe].[Klassetringruppe]': 'klassetrin',
    '[Skoleår].[Skoleår].[Skoleår]': 'skoleår',
    '[Trivselsindikator].[Trivselsindikator].[Trivselsindikator]': 'trivselsindikator',
    '[Trivselsindikator Svarintervaller].[Trivselsindikator Svarintervaller].[Trivselsindikator Svarintervaller]': 'svarinterval',
    'Andel Indikatorsvar': 'andel',
    '[Forældres Højest Fuldførte Uddannelse].[Uddannelsesgruppe].[Uddannelsesgruppe]': 'højeste_uddannelse'
})

df_fravaer_time = df_fravaer_time.rename(columns={
    '[Institution].[Beliggenhedskommune].[Beliggenhedskommune]': 'kommune',
    '[Klassetrin].[Skoletrin].[Skoletrin]': 'klassetrin',
    '[Tid].[Skoleår].[Skoleår]': 'skoleår',
    'Gennemsnitligt fravær per skoleår': 'fravaer',
    '[Forældres Højest Fuldførte Uddannelse].[Uddannelsesgruppe].[Uddannelsesgruppe]': 'højeste_uddannelse'
})

df_foraeldre_time = df_foraeldre_time.rename(columns={
    '[Forældres Højest Fuldførte Uddannelse].[Uddannelsesgruppe].[Uddannelsesgruppe]': 'uddannelsesgruppe',
    '[Institution].[Beliggenhedskommune].[Beliggenhedskommune]': 'kommune',
    '[Klassetrin].[Klassetringruppe].[Klassetringruppe]': 'klassetrin',
    '[Skoleår].[Skoleår].[Skoleår]': 'skoleår',
    'Antal elever': 'antal_elever'
})

#Vi slår kategorierne med fejl og uoplyst uddannelsesniveau sammen, da det her ikke har betydning hvilken der er tale om. 
df_foraeldre_time['uddannelsesgruppe'] = df_foraeldre_time['uddannelsesgruppe'].replace('<fejl>', '<uoplyst>')


In [7]:
display( df_trivsel_time['skoleår'].unique(),
df_fravaer_time['skoleår'].unique(),
df_foraeldre_time['skoleår'].unique())

<StringArray>
['2014/2015', '2015/2016', '2016/2017', '2017/2018', '2018/2019', '2019/2020',
 '2020/2021', '2021/2022', '2022/2023', '2023/2024', '2024/2025', '2025/2026']
Length: 12, dtype: str

<StringArray>
['2019/2020', '2020/2021', '2021/2022', '2022/2023', '2023/2024', '2024/2025']
Length: 6, dtype: str

<StringArray>
['2018/2019', '2022/2023', '2019/2020', '2021/2022', '2023/2024', '2024/2025',
 '2020/2021', '2025/2026', '2014/2015', '2017/2018', '2016/2017', '2015/2016']
Length: 12, dtype: str

In [8]:
df_foraeldre_time['antal_elever'] = (df_foraeldre_time['antal_elever'].astype(str)
                                  .str.replace('.', '', regex=False)
                                  .astype(int))

df_foraeldre_time['antal_elever'].sum()

np.int64(5565522)

In [9]:
display(df_trivsel_time['klassetrin'].unique())
display(df_fravaer_time['klassetrin'].unique())
display(df_foraeldre_time['klassetrin'].unique())

<StringArray>
['Udskoling', 'Mellemtrin']
Length: 2, dtype: str

<StringArray>
['10. klasse mv.', 'Indskoling', 'Mellemtrin', 'Udskoling']
Length: 4, dtype: str

<StringArray>
['Indskoling', 'Mellemtrin', 'Udskoling', '<uoplyst>']
Length: 4, dtype: str

In [10]:
merged_df_time = df_trivsel_time.merge(df_fravaer_time, on=['kommune', 'klassetrin', 'skoleår'], how='inner')

#Andel og fravær er tekststrenge som her konverteres til float
merged_df_time['andel_pct'] = to_float(merged_df_time['andel'])
merged_df_time['fravaer_pct'] = to_float(merged_df_time['fravaer'])

#Slet de andre kolonner med fravær og andel
merged_df_time = merged_df_time.drop(columns=["andel","fravaer"])

merged_df_time.head()

,kommune,klassetrin,skoleår,trivselsindikator,svarinterval,andel_pct,fravaer_pct
0,Albertslund,Udskoling,2019/2020,Generel trivsel,"4,1 til 5",21.2,6.8
1,Albertslund,Udskoling,2019/2020,Generel trivsel,"3,1 til 4",65.3,6.8
2,Albertslund,Udskoling,2019/2020,Generel trivsel,"2,1 til 3",12.8,6.8
3,Albertslund,Udskoling,2019/2020,Generel trivsel,1 til 2,0.7,6.8
4,Albertslund,Udskoling,2020/2021,Generel trivsel,"4,1 til 5",20.2,7.6


In [11]:
midpoints = {
    '1 til 2': 1.5,
    '2,1 til 3': 2.55,
    '3,1 til 4': 3.55,
    '4,1 til 5': 4.55
}

merged_df_time['midpoint'] = merged_df_time['svarinterval'].map(midpoints)
merged_df_time['weighted'] = merged_df_time['andel_pct'] * merged_df_time['midpoint']

trivsel_score_time = (merged_df_time.groupby(['kommune', 'klassetrin', 'skoleår', 'fravaer_pct'])
                    .apply(lambda g: g['weighted'].sum() / g['andel_pct'].sum())
                    .reset_index(name='trivsel_score'))

trivsel_score_time.head()

,kommune,klassetrin,skoleår,fravaer_pct,trivsel_score
0,Aabenraa,Mellemtrin,2019/2020,3.9,3.798601
1,Aabenraa,Mellemtrin,2020/2021,4.1,3.810850
2,Aabenraa,Mellemtrin,2021/2022,7.2,3.771528
3,Aabenraa,Mellemtrin,2022/2023,6.2,3.665700
4,Aabenraa,Mellemtrin,2023/2024,6.3,3.648550


### Forældres uddannelsesniveau

In [12]:
uddannelsesniveau = {
    '0.-9. klasse': 2,
    '10. klasse mv': 3,
    'Erhvervsuddannelse': 4,
    'Gymnasial uddannelse': 4,
    'Korte videregående uddannelser': 5,
    'Mellemlange videregående uddannelser': 6,
    'Lange videregående uddannelser': 7,
    '<uoplyst>': None  # excluded from weighted average
}

df_foraeldre_time['niveau'] = df_foraeldre_time['uddannelsesgruppe'].map(uddannelsesniveau)

kendt = df_foraeldre_time.dropna(subset=['niveau']).copy()
kendt['weighted'] = kendt['antal_elever'] * kendt['niveau']

uddannelse_score_time = (kendt.groupby(['kommune', 'klassetrin', 'skoleår'])
                    .apply(lambda g: g['weighted'].sum() / g['antal_elever'].sum())
                    .reset_index(name='foraeldre_uddannelsesniveau'))

uddannelse_score_time.head()

,kommune,klassetrin,skoleår,foraeldre_uddannelsesniveau
0,Aabenraa,Indskoling,2014/2015,4.914721
1,Aabenraa,Indskoling,2015/2016,4.887082
2,Aabenraa,Indskoling,2016/2017,4.887250
3,Aabenraa,Indskoling,2017/2018,4.837280
4,Aabenraa,Indskoling,2018/2019,4.855788


In [13]:
print(df_trivsel_time.shape)
print(df_fravaer_time.shape)
print(trivsel_score_time.shape)
print(uddannelse_score_time.shape)

(9238, 6)
(2311, 4)
(1174, 5)
(3525, 4)


In [15]:
#Eksportér den rensede data til csv-fil
trivsel_score_time.to_csv("../data/processed/trivsel_fravaer_time_merged.csv", index=False)
uddannelse_score_time.to_csv("../data/processed/uddannelse_score_time.csv", index=False)